In [ ]:
import pybamm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import dfols
import signal
from scipy.integrate import solve_ivp
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import savgol_filter
from scipy.signal import find_peaks
from scipy import interpolate, integrate
from stopit import threading_timeoutable as timeoutable
import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from batfuns import *
plt.rcParams = set_rc_params(plt.rcParams)
import winsound
from pybamm import exp, constants, Parameter
import pickle
eSOH_DIR = "../data/esoh_R/"
oCV_DIR = "../data/ocv/"
cyc_DIR = "../data/cycling/"
fig_DIR = "../figures/figures_fit/"
res_DIR = "../data/results_paper/"
resistance_DIR = "../data/resistance/"
%matplotlib widget

In [ ]:
parameter_values = get_parameter_values()

spm = pybamm.lithium_ion.SPM(
    {
        "SEI": "ec reaction limited",
        "loss of active material": "stress-driven",
        "lithium plating": "irreversible",
        "stress-induced diffusion": "false",
    }
)
# spm.print_parameter_info()
param=spm.param

In [ ]:
5.493e-14/1.484e-15

In [ ]:
cells = [3,6,9,12,15,18]
# cells = [12]
sno = 5
sim_des = f'cond{sno}'
# sim_des = sim_des+'_cv'
for cell in cells:
    # sim_des = sim_des+'_cv'
    cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
    print(cell_no)
    Ns = np.insert(N[1:]-1,0,0)
    eps_n_data,eps_p_data,c_rate_c,c_rate_d,dis_set,Temp,SOC_0 = init_exp(cell_no,dfe_0,spm,parameter_values)
    pybamm.set_logging_level("WARNING")
    # pybamm.set_logging_level("NOTICE")
    experiment = pybamm.Experiment(
        [
            ("Discharge at "+c_rate_d+dis_set,
            "Rest for 10 sec",
            "Charge at "+c_rate_c+" until 4.2V", 
            "Hold at 4.2V until C/100")
        ] *(dfe_0.N.iloc[-1]+50),
        termination="50% capacity",
    #     cccv_handling="ode",
    )
    par_val = {}
    # Room temp
    par_val[0] = [4.0312e-08,1.8157e-07,1.0776,2.3586e-09,-4.9170e-09,-1.4406e-09,4.60788219e-16,4.56607447e-19]
    # 2 Step: Ksei,Dsei; Then mech damage and plating; Different Weights
    par_val[1] = [6.8390e-08,8.7630e-06,1.0005,7.3310e-10,-1.1487e-06,-9.6950e-09,5.4930e-14,7.1060e-20]
    # 2 Step: Ksei,Dsei; Then mech damage and plating
    par_val[2] = [9.7380e-08,1.0247e-05,1.0,1.0169e-09,-9.5440e-07,-9.7510e-09,5.4930e-14,7.1060e-20]
    # 2 Step: Dsei; Then Ksei, mech damage and plating; Different Weights
    par_val[3] = [6.1320e-08,8.7270e-06,1.0,7.1320e-10,-1.0536e-06,-1.3341e-08,1.6054e-13,7.1060e-20]
    # 2 Step: Ksei,Dsei; Then mech damage and plating; Plating not 0 in step 1
    par_val[4] = [5.5390e-08,1.0610e-05,1.0244,1.0207e-09,-1.0298e-06,-1.1541e-08,5.4360e-14,1.6771e-19]
    # 2 Step: Ksei,Dsei; Then mech damage and plating; Different Weights + 50 cycles
    par_val[5] = [6.8390e-08,8.7630e-06,1.0005,7.3310e-10,-1.1487e-06,-9.6950e-09,5.4930e-14,7.1060e-20]

    parameter_values = get_parameter_values()
    parameter_values.update(
        {   
            "Positive electrode diffusion coefficient activation energy [J.mol-1]": 0,
            "Negative electrode diffusion coefficient activation energy [J.mol-1]": 0,
            "Positive electrode reference exchange-current density activation energy [J.mol-1]": 0,
            "Negative electrode reference exchange-current density activation energy [J.mol-1]": 0,
            "Positive electrode diffusion coefficient [m2.s-1]": 8e-15*0.93,
            "Negative electrode diffusion coefficient [m2.s-1]": 8e-14*30.2,
            "Positive electrode reference exchange-current density [A.m-2(m3.mol)1.5]": 3.377e-06*3.3,
            "Negative electrode reference exchange-current density [A.m-2(m3.mol)1.5]":	3.183e-06*0.62,
            "Negative electrode active material volume fraction": eps_n_data,
            "Positive electrode active material volume fraction": eps_p_data,
            "Initial temperature [K]": 273.15+Temp,
            "Ambient temperature [K]": 273.15+Temp,
            "Positive electrode LAM constant proportional term [s-1]": par_val[sno][0],
            "Negative electrode LAM constant proportional term [s-1]": par_val[sno][1],
            "Positive electrode LAM constant proportional term 2 [s-1]": par_val[sno][5],
            "Negative electrode LAM constant proportional term 2 [s-1]": par_val[sno][4],
            "Positive electrode LAM constant exponential term": par_val[sno][2],
            "Negative electrode LAM constant exponential term": par_val[sno][2],
            "SEI kinetic rate constant [m.s-1]":  par_val[sno][6], #1.08494281e-16 , 
            "EC diffusivity [m2.s-1]": par_val[sno][7],#8.30909086e-19,
            "SEI growth activation energy [J.mol-1]": 1.87422275e+04,#1.58777981e+04,
            "Lithium plating kinetic rate constant [m.s-1]": par_val[sno][3],
            "Li plating resistivity [Ohm.m]": 30000,
            "Initial inner SEI thickness [m]": 0e-09,
            "Initial outer SEI thickness [m]": 5e-09,
            "SEI resistivity [Ohm.m]": 30000.0,
            "Negative electrode partial molar volume [m3.mol-1]": 7e-06,
            "Negative electrode LAM min stress [Pa]": 0,
            "Negative electrode LAM max stress [Pa]": 0,
            "Positive electrode LAM min stress [Pa]": 0,
            "Positive electrode LAM max stress [Pa]": 0,
            # "Negative electrode diffusion coefficient [m2.s-1]": 8e-14,
            # "Positive electrode diffusion coefficient [m2.s-1]": 8e-15,
            # "Negative electrode critical stress [Pa]": 20e+06,
            # "Positive electrode critical stress [Pa]": 40e+06,
        },
        check_already_exists=False,
    )
    if cell == 15 or cell == 18:
        parameter_values.update(
            {
                "Negative electrode partial molar volume [m3.mol-1]":	0.747*7e-06,
            },
            check_already_exists=False,
        )
    all_sumvars_dict = cycle_adaptive_simulation_V2(spm, parameter_values, experiment,SOC_0, save_at_cycles=1)
    with open(res_DIR+'fast_sim_'+sim_des+"_cell_"+cell_no+'_sum_var.pickle', 'wb') as handle:
        pickle.dump(all_sumvars_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)
    plotc(all_sumvars_dict,dfe_0);
    plt.savefig(fig_DIR +'fast_sim_'+sim_des+'_'+cell_no+'_eSOH.png')
    

In [ ]:
for i in [1,4,0,5,2,3,6,7]:
    print(par_val[12][i]/par_val[0][i])

In [ ]:
dfgdf

In [ ]:
cells = [3]
sno = 11
sim_des = f'cond{sno}'
fig, ax = plt.subplots(1,1,figsize=(6,4))
i = 0
markers = ["o","v","^","1","*","d","p"]
colors = ["tab:green","tab:red","tab:green","tab:red","tab:purple","tab:brown","tab:cyan"]
for cell in cells:    
    cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
    Ns = np.insert(N_0[1:]-1,0,0)
    with open(res_DIR+'fast_sim_'+sim_des+"_cell_"+cell_no+'_sum_var.pickle', 'rb') as handle:
        df = pickle.load(handle)
    ax.plot(np.arange(0,104,1),1e9*df["X-averaged SEI thickness [m]"],color=colors[i],linewidth=2)
    i+=1
ax.set_xlabel('Ah Throughput')
ax.set_ylabel(r"Thickness [nm]")
ax.set_title(r'SEI Thickness')
# ax.set_ylim([60,102])
# ax.set_xlim([-5,3500])
fig.legend(['Room','Hot'], 
            loc="lower center",bbox_to_anchor=[0.5,-0.10], ncol=4, fontsize=11)
# plt.savefig(fig_DIR +'cycling_aging_hot_cap_1.png')
plt.show()

In [ ]:
sdfsd

In [ ]:
cell = 4
cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
print(cell_no)
sno = 0
sim_des = f'cond{sno}'
Ns = np.insert(N[1:]-1,0,0)
eps_n_data,eps_p_data,c_rate_c,c_rate_d,dis_set,Temp,SOC_0 = init_exp(cell_no,dfe,spm,parameter_values)
pybamm.set_logging_level("WARNING")
# pybamm.set_logging_level("NOTICE")
experiment = pybamm.Experiment(
    [
        ("Discharge at "+c_rate_d+dis_set,
        "Rest for 10 sec",
        "Charge at "+c_rate_c+" until 4.2V", 
        "Hold at 4.2V until C/100")
    ] *dfe.N.iloc[-1],
    # ] *40,
    termination="50% capacity",
#     cccv_handling="ode",
)
print(c_rate_c)
print(c_rate_d)
par_val = {}
# Room temp
par_val[0] = [4.0312e-08,1.8157e-07,1.0776,2.3586e-09,-4.9170e-09,-1.4406e-09]
# First Tuning
par_val[1] = [5.6076e-08,6.7429e-07,1.02,1.4576e-08,-1.7447e-07,-2.5257e-08]
# Contraint 1/2 and 2
par_val[2] = [8.0624e-08,3.6314e-07,1.02,4.7172e-09,-9.8340e-09,-2.8812e-09]
# Constraint 1/5 and 5
par_val[3] = [9.8220e-08,9.0785e-07,1.02,1.1793e-08,-2.4585e-08,-7.2030e-09]
# Constraint 1/5 and 5 with no constraints on beta2'
par_val[4] = [6.2621e-08,6.8771e-07,1.02,1.1793e-08,-1.7385e-07,-2.4901e-08]
# Constraint 1/2 and 2 with no constraints on beta2'
par_val[5] = [8.0624e-08,3.6314e-07,1.002,4.7172e-09,-2.4217e-07,-2.5212e-08]
parameter_values = get_parameter_values()
Temp = 25
parameter_values.update(
    {
        "Negative electrode active material volume fraction": eps_n_data,
        "Positive electrode active material volume fraction": eps_p_data,
        "Initial temperature [K]": 273.15+Temp,
        "Ambient temperature [K]": 273.15+Temp,
        "Positive electrode LAM constant proportional term [s-1]": par_val[sno][0],
        "Negative electrode LAM constant proportional term [s-1]": par_val[sno][1],
        "Positive electrode LAM constant proportional term 2 [s-1]": par_val[sno][5],
        "Negative electrode LAM constant proportional term 2 [s-1]": par_val[sno][4],
        "Positive electrode LAM constant exponential term": par_val[sno][2],
        "Negative electrode LAM constant exponential term": par_val[sno][2],
        "SEI kinetic rate constant [m.s-1]":  4.60788219e-16, #1.08494281e-16 , 
        "EC diffusivity [m2.s-1]": 4.56607447e-19,#8.30909086e-19,
        "SEI growth activation energy [J.mol-1]": 1.87422275e+04,#1.58777981e+04,
        "Lithium plating kinetic rate constant [m.s-1]": par_val[sno][3],
        "Initial inner SEI thickness [m]": 0e-09,
        "Initial outer SEI thickness [m]": 5e-09,
        "SEI resistivity [Ohm.m]": 30000.0,
        "Negative electrode partial molar volume [m3.mol-1]": 7e-06,
        "Negative electrode LAM min stress [Pa]": 0,
        "Negative electrode LAM max stress [Pa]": 0,
        "Positive electrode LAM min stress [Pa]": 0,
        "Positive electrode LAM max stress [Pa]": 0,
        "Negative electrode diffusion coefficient [m2.s-1]": 8e-14,
        "Positive electrode diffusion coefficient [m2.s-1]": 8e-15,
        # "Negative electrode critical stress [Pa]": 20e+06,
        # "Positive electrode critical stress [Pa]": 40e+06,
    },
    check_already_exists=False,
)
if cell == 15 or cell == 18:
    parameter_values.update(
        {
            "Negative electrode partial molar volume [m3.mol-1]":	0.747*7e-06,
        },
        check_already_exists=False,
    )

In [ ]:
experiment = pybamm.Experiment(
    [
        ("Charge at "+c_rate_c+" until 4.2V", 
        "Hold at 4.2V until C/100",
        "Discharge at "+c_rate_d+dis_set,)
    ],
    # ] *40,
    termination="50% capacity",
#     cccv_handling="ode",
)

In [ ]:
sim_long = pybamm.Simulation(spm, experiment=experiment, parameter_values=parameter_values, 
                            solver=pybamm.CasadiSolver("safe"))
sol1 = sim_long.solve(initial_soc=0.01)

In [ ]:
Temp = 45
parameter_values.update(
    {
        "Initial temperature [K]": 273.15+Temp,
        "Ambient temperature [K]": 273.15+Temp,
    },
    check_already_exists=False,
)
sim_long = pybamm.Simulation(spm, experiment=experiment, parameter_values=parameter_values, 
                            solver=pybamm.CasadiSolver("safe"))
sol2 = sim_long.solve(initial_soc=0.01)

In [ ]:
t1 = sol1["Time [s]"].entries
I1 = sol1["Current [A]"].entries
Q1 = -sol1['Discharge capacity [A.h]'].entries
Vt1 = sol1["Terminal voltage [V]"].entries

In [ ]:
t2 = sol2["Time [s]"].entries
I2 = sol2["Current [A]"].entries
Q2 = -sol2['Discharge capacity [A.h]'].entries
Vt2 = sol2["Terminal voltage [V]"].entries

In [ ]:
fig, ax = plt.subplots(1,1)
ax.plot(t1,Vt1,'g')
ax.plot(t2,Vt2,'r')
ax.legend(["Room","Hot"])
ax.set_ylabel(r"$V_t [V]$")
ax.set_xlabel("Time [s]")
ax.set_title("1.5C Room vs Hot BOL Voltage")
fig.tight_layout()
plt.savefig(fig_DIR +'hot_room_BOL_volt_comp_'+cell_no+'.png')

In [ ]:
asdasdas

In [ ]:
cell = 4
sno = 0
fig, ax = plt.subplots(1,1,figsize=(6,4))
sim_des = f'cond{sno}'
with open(res_DIR+'fast_sim_'+sim_des+"_cell_"+cell_no+'_sum_var_room.pickle', 'rb') as handle:
    df = pickle.load(handle)
ax.plot(dfe["Ah_th"],1e9*df["X-averaged SEI thickness [m]"][Ns],'g')
sim_des = f'cond{sno}'
with open(res_DIR+'fast_sim_'+sim_des+"_cell_"+cell_no+'_sum_var_hot.pickle', 'rb') as handle:
    df = pickle.load(handle)
ax.plot(dfe["Ah_th"],1e9*df["X-averaged SEI thickness [m]"][Ns],'r')
ax.set_xlabel('Ah Throughput')
ax.set_ylabel(r"Thickness [nm]")
ax.set_title(r'SEI Thickness')
# ax.set_ylim([60,102])
# ax.set_xlim([-5,3500])
fig.legend(['Room','Hot'], 
            loc="lower center",bbox_to_anchor=[0.5,-0.10], ncol=4, fontsize=11)
plt.savefig(fig_DIR +'sei_hot_room_4.png')
plt.show()

In [ ]:
cell = 1
sno = 0
fig, ax = plt.subplots(1,1,figsize=(6,4))
sim_des = f'cond{sno}'
with open(res_DIR+'fast_sim_'+sim_des+"_cell_"+cell_no+'_sum_var_room.pickle', 'rb') as handle:
    df = pickle.load(handle)
ax.plot(df["Cycle number"],df["Loss of lithium to SEI [mol]"],'g')
sim_des = f'cond{sno}'
with open(res_DIR+'fast_sim_'+sim_des+"_cell_"+cell_no+'_sum_var_hot.pickle', 'rb') as handle:
    df = pickle.load(handle)
ax.plot(df["Cycle number"],df["Loss of lithium to SEI [mol]"],'r')
ax.set_xlabel('Cycle Number')
ax.set_ylabel(r"LLI [mol]")
ax.set_title(r'LLI due to SEI')
# ax.set_ylim([60,102])
# ax.set_xlim([-5,3500])
fig.legend(['Room','Hot'], 
            loc="lower center",bbox_to_anchor=[0.5,-0.10], ncol=4, fontsize=11)
plt.savefig(fig_DIR +'lli_sei_hot_room_1.png')
plt.show()

In [ ]:
cell = 1
sno = 0
sim_des = f'cond{sno}'
# fig, ax = plt.subplots(1,3,figsize=(10,4),sharex=True)
fig, ax = plt.subplots(1,3,figsize=(10,4),sharex=True)
with open(res_DIR+'fast_sim_'+sim_des+"_cell_"+cell_no+'_sum_var_room.pickle', 'rb') as handle:
    df1 = pickle.load(handle)
with open(res_DIR+'fast_sim_'+sim_des+"_cell_"+cell_no+'_sum_var_hot.pickle', 'rb') as handle:
    df2 = pickle.load(handle)
ax1 = ax.flat[0]
ax1.plot(df1["Cycle number"],(df1["C_n"][0]-df1["C_n"])/df1["C_n"][0]*100,color='green',linewidth=3)
ax1.plot(df2["Cycle number"],(df2["C_n"][0]-df2["C_n"])/df2["C_n"][0]*100,color='red',linewidth=3)
ax2 = ax.flat[1]
ax2.plot(df1["Cycle number"],(df1["C_p"][0]-df1["C_p"])/df1["C_n"][0]*100,color='green',linewidth=3)
ax2.plot(df2["Cycle number"],(df2["C_p"][0]-df2["C_p"])/df2["C_n"][0]*100,color='red',linewidth=3)
ax3 = ax.flat[2]
ax3.plot(df1["Cycle number"],df1["Loss of lithium inventory [%]"],color='green',linewidth=3)
ax3.plot(df2["Cycle number"],df2["Loss of lithium inventory [%]"],color='red',linewidth=3)
ax3.set_xlabel('Cycle Number')
ax1.set_ylabel(r"Loss [%]")
ax2.set_ylabel(r"Loss [%]")
ax3.set_ylabel(r"Loss [%]")
ax1.set_title(r'$LAM_{NE}$')
ax2.set_title(r'$LAM_{PE}$')
ax3.set_title(r'$LLI$')
# ax.set_ylim([60,102])
# ax1.set_xlim([-5,3500])
# ax.legend(['C/5','1.5C','2C','Mixed Crate','C/5 50% DOD','Mixed 50% DOD','Drive Cycle'])
fig.legend(['Room','Hot'], 
            loc="lower center",bbox_to_anchor=[0.5,-0.05], ncol=2, fontsize=11)
fig.tight_layout()
plt.savefig(fig_DIR +'cycling_aging_room_hot_deg_mode_1.png')
plt.show()

In [ ]:
keys = df.keys()
keys = list(keys)

In [ ]:

ax1 = ax.flat[0]
if cell == 7:
    ax1.plot(dfe["Ah_th"],(df["C_n"][0]-df["C_n"][Ns])/df["C_n"][0]*100,color=colors[i],linewidth=3)
else:
    ax1.plot(dfe["Ah_th"],(df["C_n"][0]-df["C_n"][Ns])/df["C_n"][0]*100,'--',color=colors[i],linewidth=3)
ax2 = ax.flat[1]
if cell == 7:
    ax2.plot(dfe["Ah_th"],(df["C_p"][0]-df["C_p"][Ns])/df["C_n"][0]*100,color=colors[i],linewidth=3)
else:
    ax2.plot(dfe["Ah_th"],(df["C_p"][0]-df["C_p"][Ns])/df["C_n"][0]*100,'--',color=colors[i],linewidth=3)
ax3 = ax.flat[2]
if cell == 7:
    ax3.plot(dfe["Ah_th"],df["Loss of lithium inventory [%]"][Ns],color=colors[i],linewidth=3)
else:
    ax3.plot(dfe["Ah_th"],df["Loss of lithium inventory [%]"][Ns],'--',color=colors[i],linewidth=3)

# ax1.set_xlabel('Ah Throughput')
# ax2.set_xlabel('Ah Throughput')
ax3.set_xlabel('Ah Throughput')
ax1.set_ylabel(r"Loss [%]")
ax2.set_ylabel(r"Loss [%]")
ax3.set_ylabel(r"Loss [%]")
ax1.set_title(r'$LAM_{NE}$')
ax2.set_title(r'$LAM_{PE}$')
ax3.set_title(r'$LLI$')
# ax.set_ylim([60,102])
ax1.set_xlim([-5,3500])
# ax.legend(['C/5','1.5C','2C','Mixed Crate','C/5 50% DOD','Mixed 50% DOD','Drive Cycle'])
fig.legend(['C/5','1.5C','2C','Ch:C/5, Dh:1.5C'], 
            loc="lower center",bbox_to_anchor=[0.5,-0.05], ncol=2, fontsize=11)
fig.tight_layout()
plt.savefig(fig_DIR +'cycling_aging_room_deg_mode_1.png')
plt.show()